# Chapter 8 &mdash; Designing a Regular Expression, and Kleene's Theorem

**Concept 3 of the Chapter 8 decomposition:** *Designing a Regular Expression Systematically, and Kleene's Theorem*

Decompose into salient patterns, generalize each to allow irrelevant symbols, then combine &mdash; and RE = NFA = DFA.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8-RE/Concept-Designing-RE-And-Kleene/Concept-Designing-RE-And-Kleene.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_RE2NFA     import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


A reliable recipe for designing an RE:

1. **decompose** the language into salient patterns;
2. **generalize** each pattern to allow the irrelevant symbols around it &mdash; usually by
   sprinkling $(0+1)^*$;
3. **combine** with $+$;
4. **cross-check** against a DFA you designed independently.

> **Kleene's Theorem.** A language is regular iff it is denoted by some regular
> expression.

With Chapter 7's theorem, the chain closes: **RE = NFA = DFA**. Every conversion in
this chapter and the next is a constructive proof of one link.

## 2. Definitions

### Step 1-3 on a worked example

$L$ = strings containing `01` **or** ending in `11`.

In [ ]:
def re_dfa(r): return min_dfa(nfa2dfa(re2nfa(r)))

pat1 = "(0+1)*01(0+1)*"     # contains 01 -- generalized on both sides
pat2 = "(0+1)*11"           # ends in 11  -- generalized on the left only
whole = "(%s)+(%s)" % (pat1, pat2)
print("RE :", whole)

### Step 4: an independent DFA for the same language

In [ ]:
D = md2mc('''DFA
I   : 0 -> S0        !! seen a 0, no pattern yet
I   : 1 -> S1        !! seen one trailing 1
S0  : 0 -> S0
S0  : 1 -> Fok       !! '01' found -- accept forever
S1  : 0 -> S0
S1  : 1 -> Fend      !! ends in '11'
Fend: 1 -> Fend
Fend: 0 -> S0
Fok : 0 | 1 -> Fok
''')

<!-- nav-strip -->

---

&larr;&nbsp;[Ch8&nbsp;2.&nbsp;RE to NFA: the Thompson-Style Constructions for $\varepsilon$, $a$, Concatenation, Union and Star](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8-RE/Concept-Thompson-Constructions/Concept-Thompson-Constructions.ipynb) &nbsp;&middot;&nbsp; [**Chapter 8** index](https://github.com/ganeshutah/Jove/blob/master/Chapter8-RE/README.md) &nbsp;&middot;&nbsp; [Ch8&nbsp;4.&nbsp;Regular Expressions Are Error-Prone, and That Is a Security Problem](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter8-RE/Concept-RE-Are-Error-Prone/Concept-RE-Are-Error-Prone.ipynb)&nbsp;&rarr;

---

## 3. Tests

The two designs agree &mdash; the cross-check of step 4.

In [ ]:
R = re_dfa(whole)
print("RE-derived DFA : %d states" % len(R["Q"]))
print("hand DFA (min) : %d states" % len(min_dfa(D)["Q"]))
print("langeq :", langeq_dfa(R, min_dfa(D)))
print("iso    :", iso_dfa(R, min_dfa(D)))
assert langeq_dfa(R, min_dfa(D)) and iso_dfa(R, min_dfa(D))

Both match the specification.

In [ ]:
from itertools import product
spec = lambda s: ('01' in s) or s.endswith('11')
strs = [''.join(p) for k in range(11) for p in product('01', repeat=k)]
assert all(accepts_dfa(R, s) == spec(s) for s in strs)
assert all(accepts_dfa(D, s) == spec(s) for s in strs)
print("both agree with the spec on all %d strings up to length 10" % len(strs))

Step 2 matters: forgetting to generalize gives the wrong language.

In [ ]:
naive = "01+11"
print("L(01+11) up to length 4 :",
      [s for s in strs if len(s) <= 4 and accepts_dfa(re_dfa(naive), s)])
assert not langeq_dfa(re_dfa(naive), R)
print("\nWithout the (0+1)* sprinkles the RE only matches the patterns exactly.")

**Kleene's theorem**, exercised: RE &rarr; NFA &rarr; DFA &rarr; minimal DFA, language preserved.

In [ ]:
N = re2nfa(whole)
chain = [("RE -> NFA", N), ("-> DFA", nfa2dfa(N)), ("-> minimal", min_dfa(nfa2dfa(N)))]
for name, X in chain:
    acc = accepts_nfa if "Q0" in X else accepts_dfa
    assert all(acc(X, s) == spec(s) for s in strs)
    print("%-12s |Q| = %3d  language preserved" % (name, len(X["Q"])))

## 4. Animation

The cross-checked machine, from either design.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(re_dfa(whole), FuseEdges=True)

## 5. Exercises


1. Design an RE for "even number of 0s". Cross-check against Chapter 5's DFA.
2. Which step of the recipe do beginners skip most often?
3. State Kleene's theorem and name the two constructions that prove it.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 255 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter8-RE/Concept-Designing-RE-And-Kleene')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')